# Publicación MQTT a InfluxDB — del CSV al broker en velocidad acelerada

> _Tutorial · Caso de uso: **A — Pipeline IoT CENTINELA+** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/03-architecture-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Tomar el mock In-Gauge de AULA01 (1 semana × 1 minuto) y publicarlo vía MQTT con topic canónico, simulando los sensores reales de CENTINELA+. Comprobar que cada mensaje aterriza en `captia_point`.


## 2. Qué se aprende

- Cómo usar `paho-mqtt` para publicar en Mosquitto.
- Estructura del payload `{value, ts_ns}`.
- Velocidad acelerada vs tiempo real.
- Cómo verificar la llegada con una query Flux.


## 3. Contexto del caso de uso

Los datasets públicos tienen resolución de minutos o segundos. Esperar a que pase el tiempo real sería absurdo para una clase. La técnica habitual es **publicar tan rápido como permita el broker**, pero conservando los timestamps originales del dataset.


## 4. Relación con CENTINELA+

Reproducimos el papel del firmware del sensor. La diferencia es que publicamos a velocidad acelerada para no esperar 7 días de clase.


## 5. Relación con Medallion

Capa **bronce** (CSV In-Gauge) → **plata** (`captia_point` en InfluxDB).


## 6. Datos de entrada

`notebooks/_data/ingauge_aula01_mock.csv` (1 semana × 1min, 9 columnas).


## 7. Schema CAPTIA esperado

Para cada fila del CSV producimos varios topics MQTT (uno por variable). Mapping In-Gauge → CAPTIA según `docs/specs/synthetic-bms/02-domain-spec.md`.


In [ ]:
mapping_ingauge = {
    "Indoor_CO2": "co2",
    "Indoor_Temp": "temperature_01",
    "Indoor_Hum": "relative_humidity_01",
    "Indoor_Noise": "avg_sound_level",
    "Indoor_Lux": "luminosity",
    "People_Count": "people_count",
    "Occupied": "occupancy",
    "CoolingState": "ac_state",
}
mapping_ingauge


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.

Si el stack está levantado, importamos `paho.mqtt.client`. Si no, definimos un cliente mock que registra los mensajes en memoria.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Cargamos el CSV mock (con la cabecera `# MOCK ...`) y mostramos las primeras filas.


In [ ]:
csv_path = ROOT / "notebooks" / "_data" / "ingauge_aula01_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df.head()


## 10. Exploración paso a paso

Resumen estadístico del dataset y conteo por hora.


In [ ]:
print("Filas:", len(df), "  Variables CSV:", len(df.columns) - 1)
df.set_index("timestamp").resample("1h").mean(numeric_only=True).head()


## 11. Transformación bronce → plata

Construimos los mensajes MQTT (topic + payload) que vamos a publicar. Si tenemos `paho-mqtt` y el broker funciona, publicamos; si no, los ponemos en una lista en memoria que demuestra el flujo.


In [ ]:
def iter_mqtt_messages(df, asset="AULA01", env="dev", tenant="default", site="ies_simarro"):
    for _, row in df.iterrows():
        ts_ns = int(pd.Timestamp(row["timestamp"]).value)
        for csv_col, captia_var in mapping_ingauge.items():
            if csv_col not in row or pd.isna(row[csv_col]):
                continue
            topic = build_topic(env=env, tenant=tenant, site=site,
                                 asset=asset, variable=captia_var)
            payload = {"value": float(row[csv_col]), "ts_ns": ts_ns}
            yield topic, payload

# Para clase: tomamos solo los primeros 200 mensajes (~25 filas) para no saturar
muestras = list(iter_mqtt_messages(df.head(25)))
print(f"Generados {len(muestras)} mensajes MQTT (primeros 25 minutos del mock)")
muestras[0]


## 12. Construcción de capa oro

No aplica para este notebook (capa plata es el objetivo).


## 13. Visualizaciones explicativas

Distribución de mensajes por variable (debe ser uniforme: 8 variables × 25 filas = 200).


In [ ]:
import collections
counts = collections.Counter(t.split("/")[-1] for t, _ in muestras)
pd.Series(counts).sort_values().plot.barh(color="#3F51B5", figsize=(7, 3))
plt.title("Mensajes MQTT por variable")
plt.tight_layout()


## 14. Validaciones

Comprobamos que: cada topic tiene 6 niveles, el payload tiene los 2 campos requeridos, los timestamps son monotónicos por variable.


In [ ]:
import json

assert all(t.count("/") == 6 for t, _ in muestras)
assert all(set(p.keys()) == {"value", "ts_ns"} for _, p in muestras)

# Monotonicidad por (topic)
prev_ts = {}
for topic, payload in muestras:
    if topic in prev_ts:
        assert payload["ts_ns"] >= prev_ts[topic]
    prev_ts[topic] = payload["ts_ns"]
print("Validaciones OK · ejemplos:")
for topic, payload in muestras[:3]:
    print(f"  {topic} → {json.dumps(payload)}")


## 15. Errores comunes

1. **No esperar entre `connect` y `publish`** — el cliente puede no haber completado el handshake.
2. **Publicar a 1 Hz pero el broker rechaza** — confirmar QoS 1 y configurar `max_inflight_messages` en el cliente.
3. **Olvidar `client.loop_start()`** — sin loop, los ACKs no se procesan.
4. **Usar `time.sleep(60)` para simular tiempo real** — la clase dura 50 minutos, no 7 días.
5. **Confundir `value` con `value_str`** — InfluxDB rechaza tipos mixtos.


## 16. Ejercicios propuestos

1. Añade `valve_state` al mapping y publícala.
2. Implementa `publish_with_backpressure(client, msgs, target_rate)` que envía a `target_rate` msgs/s usando `time.sleep`.
3. Simula una pérdida del 5% de mensajes y mide el impacto.


## 17. Cómo se reutiliza con datos reales

Para enviar telemetría real basta con cambiar `MQTT_HOST` en `.env` y leer del topic real. La función `iter_mqtt_messages` se reusa palabra por palabra; solo cambia el origen de los datos.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `01_case_A_pipeline_iot/03_validacion_telegraf_influx_grafana.ipynb`.
- Documento web del caso: `docs/contracts/mqtt-topics.md`.
